# 01 — 2026 data audit
Reproducible audit of the committed August 2026 Tashkent apartment listing snapshot. The target is advertised asking price, not completed sale price.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
df = pd.read_csv(ROOT / 'data' / 'apartment_listings_2026.csv')
df.head()

## Shape, schema, missingness, and duplicates
These checks establish whether the file is suitable before any model decision.

In [ ]:
fingerprint = ['district', 'rooms', 'size_m2', 'level', 'max_levels', 'is_new_building']
print('shape:', df.shape)
display(df.dtypes.rename('dtype').to_frame())
display(df.isna().sum().rename('missing').to_frame())
print('feature + target duplicates:', df.duplicated(fingerprint + ['listing_price_usd']).sum())
print('identical-feature extra rows:', df.duplicated(fingerprint).sum())
print('invalid floors:', (df['level'] > df['max_levels']).sum())
print('listing dates:', df['listing_date'].min(), 'to', df['listing_date'].max())

**Observation:** The parsed snapshot has no missing required fields. `load_dataset` removes 257 obvious invalid/category/currency/unit rows and 396 exact feature + target duplicates. Identical feature fingerprints are grouped during holdout and CV to reduce relisting leakage.

In [ ]:
display(df.describe(include='all').T)
display(df['district'].value_counts().rename('rows').to_frame())

**Observation:** District coverage is uneven and Yangihayot is absent. Splits are district-stratified at feature-group level, and every district metric is reported with its test count.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df['listing_price_usd'], bins=60, ax=axes[0])
axes[0].set_title('Advertised asking-price distribution (USD)')
sns.scatterplot(data=df, x='size_m2', y='listing_price_usd', hue='district', legend=False, alpha=.35, ax=axes[1])
axes[1].set_title('Asking price vs apartment size')
plt.tight_layout()

**Observation:** Asking price is right-skewed. Conservative validity limits remove obvious errors while retaining luxury listings; MAE is primary and RMSE exposes the luxury tail. Price per m² is excluded because it contains the target.

In [ ]:
from src.data import load_dataset
X, y, audit = load_dataset(ROOT / 'data' / 'apartment_listings_2026.csv')
audit